# 实践 10.3  共识峰、重复一致性与差异峰的联合判定

《统计遗传与智能育种学》第十章 · 电子资源 `10c_peak_consensus_diff.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：10c_peak_consensus_diff.ipynb。实践模拟两种处理各3个生物学重复和55个真实差异峰。二元分支只保留每个样本中峰的有无状态，并在共识集合内用Fisher精确检验；计数分支在同一集合上保留强度信息，对归一化后的对数计数使用Welch检验作为简化基线。两个分支都在同一检验族内实施Benjamini–Hochberg校正。该实践用于区分重复一致性、共识特征空间和差异计数三个问题，不把Welch检验作为正式ATAC-seq计数模型。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：两个重复的峰集合Jaccard相似系数约0.826，共识峰集合含312个区域，其中覆盖47/55个真实差异峰；3+3重复下二元Fisher+BH得到0个发现，而统一计数简化检验+BH得到29个发现（29个真阳性、0个假阳性）。每组3个重复时，双侧Fisher检验的最小P值为0.10，但不代表计数检验在任何真实数据中都能达到零假阳性。

## 输入、完整代码、实际输出与解释

输入与可复现条件：两组各3个模拟生物学重复、500个候选区域、55个真实差异峰；SEED=1003；无需外部文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=1003
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1003


In [2]:
from scipy.stats import ttest_ind, fisher_exact
nrep=3; npk=500; true_diff=np.zeros(npk,bool); true_diff[:55]=True; base=rng.gamma(2,30,npk); counts=[]; binary=[]; libs=[]
for group in [0,1]:
 for rep in range(nrep):
  lib=rng.lognormal(0,.18); libs.append(lib); mu=lib*base*(1+1.8*group*true_diff); x=rng.poisson(mu); counts.append(x); threshold=np.quantile(x,.45); binary.append(x>threshold)
counts=np.array(counts); binary=np.array(binary); libs=np.array(libs); print('binary peak-set Jaccard replicate0/1',round(np.sum(binary[0]&binary[1])/np.sum(binary[0]|binary[1]),3)); universe=(binary.sum(0)>=2); print('consensus universe peaks',universe.sum(),'true DA represented',np.sum(universe&true_diff),'of',true_diff.sum())

# 二元peak出现只保留有/无；统一universe上的计数保留强度。两者都明确在universe内做BH校正。
def bh_family(p,mask):
 q=np.ones(len(p)); v=np.asarray(p)[mask]; o=np.argsort(v); z=v[o]*len(v)/np.arange(1,len(v)+1); z=np.minimum.accumulate(z[::-1])[::-1]; out=np.empty_like(z); out[o]=np.clip(z,0,1); q[mask]=out; return q
pbin=np.ones(npk); pcnt=np.ones(npk); lognorm=np.log2((counts+.5)/libs[:,None])
for j in np.where(universe)[0]:
 a=int(binary[:nrep,j].sum()); b=int(binary[nrep:,j].sum()); pbin[j]=fisher_exact([[a,nrep-a],[b,nrep-b]]).pvalue; pcnt[j]=ttest_ind(lognorm[:nrep,j],lognorm[nrep:,j],equal_var=False).pvalue
for name,pv in [('binary_presence_Fisher',pbin),('unified_counts_teaching_test',pcnt)]:
 q=bh_family(pv,universe); hit=q<.05; print(name,'discoveries',hit.sum(),'TP',np.sum(hit&true_diff),'FP',np.sum(hit&~true_diff),'recall',round(np.sum(hit&true_diff)/true_diff.sum(),3))
print('count replicate correlations group0',np.round(np.corrcoef(lognorm[:nrep]),3))


binary peak-set Jaccard replicate0/1 0.826
consensus universe peaks 312 true DA represented 47 of 55
binary_presence_Fisher discoveries 0 TP 0 FP 0 recall 0.0
unified_counts_teaching_test discoveries 29 TP 29 FP 0 recall 0.527
count replicate correlations group0 [[1.    0.94  0.93 ]
 [0.94  1.    0.933]
 [0.93  0.933 1.   ]]


结果解释：共识集合只覆盖47/55个真实差异峰，说明特征空间构建本身已经影响可检验对象。每组3个重复时，双侧Fisher检验最小P值为0.10，校正前也未检出；采用对数归一化计数的Welch检验检出29个真阳性。本次0个假阳性是模拟结果，不是方法保证；实际项目应使用与计数分布和实验设计相匹配的正式模型。

可修改练习：把每组重复数nrep从3改为6，或降低真实倍数变化，再比较二元峰有无检验与统一计数分析的召回率；同时记录统一峰集合是否遗漏真实差异区域。